[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/editorial-v2/notebooks/07_Observability_and_State_Estimation.ipynb)

# DiveLab

## Notebook 07 — Observability and State Estimation

**From measurements to hidden state**

### Guiding question

If only depth is available from measurements, can vertical velocity be reconstructed?

This notebook uses the local linear model from Chapter 6. It demonstrates estimation concepts with simplified synthetic data and is not a substitute for certified diving instrumentation or procedures.

## Learning objectives

By the end of this notebook, you should be able to:

- distinguish true state, measurement, derived output and estimate;
- construct the depth-output matrix $C$;
- calculate the observability matrix and its rank;
- explain why depth history contains velocity information;
- compare direct differentiation with model-based estimation;
- construct a Luenberger observer with correctly placed poles;
- interpret innovation and estimation error;
- investigate observer speed versus noise sensitivity.

## From Notebook 06 to Notebook 07

Notebook 06 derived the local perturbation model

$$
\dot{\boldsymbol{\xi}}=A\boldsymbol{\xi},
\qquad
\boldsymbol{\xi}=
\begin{bmatrix}
\delta z\\
\delta v
\end{bmatrix}.
$$

The state contains two variables, but the available output may contain only depth. Notebook 07 asks whether the missing velocity can be reconstructed from the output history and the model.

## Measurement vocabulary

In a physical dive computer:

- ambient pressure is **measured** by a sensor;
- depth is **derived** from pressure and environmental assumptions;
- vertical velocity may be **estimated** from depth history and a model;
- future depth is **predicted** by propagating the estimate.

For this laboratory, we treat derived depth deviation as the available measurement and combine all imperfections into additive noise.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

## Reconstruct the Chapter 6 linear model

Use the canonical operating state:

- mass $m=85\ \mathrm{kg}$;
- equilibrium depth $z^*=20\ \mathrm{m}$;
- surface flexible-gas volume $V_{g0}=8\ \mathrm{L}$.

The local coefficient is

$$
a=\frac{k_B}{m}<0,
$$

and

$$
A=
\begin{bmatrix}
0&-1\\
a&0
\end{bmatrix}.
$$

In [ ]:
rho = 1025.0          # seawater density [kg/m^3]
g = 9.80665           # gravitational acceleration [m/s^2]
p0 = 101_325.0        # surface absolute pressure [Pa]
mass = 85.0           # diver-and-equipment mass [kg]
equilibrium_depth = 20.0  # [m]
surface_gas_volume = 8.0e-3  # [m^3]

pressure_at_equilibrium = p0 + rho * g * equilibrium_depth
k_b = (
    -(rho * g) ** 2
    * surface_gas_volume
    * p0
    / pressure_at_equilibrium**2
)
a = k_b / mass

A = np.array([
    [0.0, -1.0],
    [a, 0.0],
])

print(f"Buoyancy sensitivity k_B: {k_b:.6f} N/m")
print(f"Local coefficient a:       {a:.8f} 1/s^2")
print("Plant eigenvalues [1/s]:  ", np.linalg.eigvals(A))

## Depth-only output

For the idealized depth-deviation output,

$$
y=C\boldsymbol{\xi},
\qquad
C=
\begin{bmatrix}
1&0
\end{bmatrix}.
$$

With additive noise,

$$
y_m=C\boldsymbol{\xi}+n.
$$

In [ ]:
C = np.array([[1.0, 0.0]])

print("A =")
print(A)
print("C =")
print(C)

## Physical observability intuition

Since $y=\delta z$ and $\delta\dot z=-\delta v$,

$$
\dot y=-\delta v.
$$

The time history of depth therefore carries information about the unmeasured velocity. The rank test formalizes this intuition.

## Build the observability matrix

For two states,

$$
\mathcal{O}=
\begin{bmatrix}
C\\
CA
\end{bmatrix}.
$$

In [ ]:
observability_matrix = np.vstack([C, C @ A])
observability_rank = np.linalg.matrix_rank(observability_matrix)

print("Observability matrix:")
print(observability_matrix)
print(f"Rank: {observability_rank}")
print(f"State dimension: {A.shape[0]}")

The rank equals the state dimension, so $(A,C)$ is observable. In the ideal linear model, depth history contains enough information to reconstruct both state components.

Observability is structural. It does not guarantee an accurate estimate when measurements are noisy or the model is wrong.

## Simulate the true local state

In [ ]:
sample_period = 0.01  # [s]
duration = 20.0       # [s]
sample_count = int(round(duration / sample_period)) + 1
sample_times = np.linspace(0.0, duration, sample_count)
true_initial_state = np.array([0.0, 0.05])


def linear_state_derivative(time_s, state):
    return A @ state


true_solution = solve_ivp(
    linear_state_derivative,
    (0.0, duration),
    true_initial_state,
    t_eval=sample_times,
    rtol=1e-10,
    atol=1e-12,
)

if not true_solution.success:
    raise RuntimeError(true_solution.message)

true_state = true_solution.y.T
true_output = (C @ true_solution.y).ravel()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True)

axes[0].plot(sample_times, true_state[:, 0])
axes[0].set_ylabel("Depth deviation [m]")
axes[0].set_title("True local state")
axes[0].grid(True)

axes[1].plot(sample_times, true_state[:, 1])
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Velocity deviation [m/s]")
axes[1].grid(True)

plt.tight_layout()
plt.show()

The open-loop plant is unstable, as Chapter 6 established. Observability does not require a stable plant: it asks whether the state is encoded in the output history.

## Add depth-measurement noise

In [ ]:
rng = np.random.default_rng(42)
measurement_noise_std = 0.02  # [m]
measurement_noise = rng.normal(
    0.0,
    measurement_noise_std,
    size=true_output.size,
)
measured_depth = true_output + measurement_noise

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(sample_times, true_output, label="true depth deviation")
plt.plot(
    sample_times,
    measured_depth,
    alpha=0.45,
    linewidth=0.8,
    label="noisy derived depth",
)
plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Available output")
plt.grid(True)
plt.legend()
plt.show()

## Why direct differentiation is fragile

In the noise-free model,

$$
\delta v=-\dot y.
$$

A finite difference divides sample-to-sample measurement variation by the small sampling period. Noise is therefore strongly amplified.

In [ ]:
velocity_from_difference = -np.gradient(measured_depth, sample_period)

plt.figure(figsize=(8, 4.5))
plt.plot(
    sample_times,
    velocity_from_difference,
    color="0.65",
    linewidth=0.8,
    label="velocity from noisy differentiation",
)
plt.plot(
    sample_times,
    true_state[:, 1],
    color="black",
    linewidth=2,
    label="true velocity deviation",
)
plt.xlabel("Time [s]")
plt.ylabel("Velocity deviation [m/s]")
plt.title("Direct differentiation amplifies depth noise")
plt.grid(True)
plt.legend()
plt.show()

The structural relation is correct, but this numerical implementation is poor. A model-based observer estimates the state without treating every measurement fluctuation as physical motion.

## Luenberger observer

Use

$$
\dot{\hat{\boldsymbol{\xi}}}
=
A\hat{\boldsymbol{\xi}}
+L\left(y_m-C\hat{\boldsymbol{\xi}}\right).
$$

For desired poles $p_1$ and $p_2$, the correct gain for this sign convention is

$$
l_1=-(p_1+p_2),
\qquad
l_2=a-p_1p_2.
$$

In [ ]:
def observer_gain_from_poles(pole_1, pole_2):
    l_1 = -(pole_1 + pole_2)
    l_2 = a - pole_1 * pole_2
    return np.array([[l_1], [l_2]])


desired_poles = (-0.7, -1.0)
L = observer_gain_from_poles(*desired_poles)
observer_eigenvalues = np.linalg.eigvals(A - L @ C)

print("Observer gain L:")
print(L)
print("Requested poles [1/s]:", desired_poles)
print("Achieved poles [1/s]: ", observer_eigenvalues)

The achieved eigenvalues must match the requested poles. This check is essential because a sign error in $L$ can make the estimation error unstable.

## Simulate the observer

In [ ]:
def simulate_observer(measurement, initial_estimate, gain):
    estimate = np.zeros((measurement.size, 2))
    innovation = np.zeros(measurement.size)
    estimate[0] = np.asarray(initial_estimate, dtype=float)

    for index in range(measurement.size - 1):
        predicted_output = float((C @ estimate[index])[0])
        innovation[index] = measurement[index] - predicted_output
        estimate_derivative = (
            A @ estimate[index]
            + gain[:, 0] * innovation[index]
        )
        estimate[index + 1] = (
            estimate[index]
            + sample_period * estimate_derivative
        )

    innovation[-1] = measurement[-1] - float((C @ estimate[-1])[0])
    return estimate, innovation


initial_estimate = np.array([0.30, -0.15])
estimated_state, innovation = simulate_observer(
    measured_depth,
    initial_estimate,
    L,
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True)

axes[0].plot(sample_times, true_state[:, 0], label="true depth deviation")
axes[0].plot(sample_times, estimated_state[:, 0], label="estimated depth deviation")
axes[0].set_ylabel("Depth deviation [m]")
axes[0].set_title("Luenberger state estimate")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(sample_times, true_state[:, 1], label="true velocity deviation")
axes[1].plot(sample_times, estimated_state[:, 1], label="estimated velocity deviation")
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Velocity deviation [m/s]")
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

## Key result

The observer reconstructs velocity even though velocity is never supplied as a measurement. It succeeds because the dynamics couple velocity to the measured depth output and the pair $(A,C)$ is observable.

The observer also corrects its deliberately incorrect initial state.

## Compare velocity-estimation errors

In [ ]:
evaluation_mask = sample_times >= 5.0

difference_rmse = np.sqrt(np.mean(
    (velocity_from_difference[evaluation_mask] - true_state[evaluation_mask, 1]) ** 2
))
observer_rmse = np.sqrt(np.mean(
    (estimated_state[evaluation_mask, 1] - true_state[evaluation_mask, 1]) ** 2
))

print(f"Differentiation velocity RMSE: {difference_rmse:.4f} m/s")
print(f"Observer velocity RMSE:        {observer_rmse:.4f} m/s")

The numerical values depend on the assumed noise, sampling period, model and observer poles. Their purpose is to compare methods under one reproducible experiment, not to claim field performance.

## Estimation error

Define

$$
\mathbf{e}=\boldsymbol{\xi}-\hat{\boldsymbol{\xi}}.
$$

Without noise or model mismatch,

$$
\dot{\mathbf{e}}=(A-LC)\mathbf{e}.
$$

In [ ]:
estimation_error = true_state - estimated_state

fig, axes = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True)

axes[0].plot(sample_times, estimation_error[:, 0])
axes[0].set_ylabel("Depth error [m]")
axes[0].set_title("State-estimation error")
axes[0].grid(True)

axes[1].plot(sample_times, estimation_error[:, 1])
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Velocity error [m/s]")
axes[1].grid(True)

plt.tight_layout()
plt.show()

The deterministic initial-condition error decays according to the chosen observer poles. Measurement noise prevents the error from becoming identically zero and produces persistent small fluctuations.

## Innovation

The innovation

$$
r=y_m-C\hat{\boldsymbol{\xi}}
$$

is the difference between the available measurement and the output predicted by the observer.

In [ ]:
plt.figure(figsize=(8, 4.2))
plt.plot(sample_times, innovation, linewidth=0.8)
plt.axhline(0.0, color="black", linewidth=1)
plt.xlabel("Time [s]")
plt.ylabel("Depth innovation [m]")
plt.title("Measurement residual")
plt.grid(True)
plt.show()

The innovation drives state correction. It also provides diagnostic information, but systematic fault detection requires explicit noise and fault models and is postponed to later integrated-system material.

## Observer speed versus noise sensitivity

In [ ]:
observer_designs = {
    "slow poles (-0.25, -0.35)": (-0.25, -0.35),
    "nominal poles (-0.7, -1.0)": (-0.7, -1.0),
    "fast poles (-2.0, -3.0)": (-2.0, -3.0),
}

design_results = {}
for label, poles in observer_designs.items():
    gain = observer_gain_from_poles(*poles)
    estimate, residual = simulate_observer(
        measured_depth,
        initial_estimate,
        gain,
    )
    velocity_rmse = np.sqrt(np.mean(
        (estimate[evaluation_mask, 1] - true_state[evaluation_mask, 1]) ** 2
    ))
    design_results[label] = (gain, estimate, velocity_rmse)
    print(f"{label}: velocity RMSE = {velocity_rmse:.5f} m/s")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    sample_times,
    true_state[:, 1],
    color="black",
    linewidth=2.5,
    label="true velocity deviation",
)

for label, (_, estimate, _) in design_results.items():
    plt.plot(sample_times, estimate[:, 1], label=label)

plt.xlabel("Time [s]")
plt.ylabel("Velocity deviation [m/s]")
plt.title("Observer speed and measurement-noise sensitivity")
plt.grid(True)
plt.legend()
plt.show()

## Interpretation

Slow observer poles reduce sensitivity to high-frequency measurement fluctuations but take longer to remove initial-condition error. Fast poles correct the initial estimate rapidly but inject more measurement noise into the reconstructed velocity.

Pole placement controls deterministic error dynamics; it does not automatically provide the best noisy-data estimate.

## Output choice and physical coupling

If velocity alone were available,

$$
C_v=
\begin{bmatrix}
0&1
\end{bmatrix}.
$$

The corresponding observability matrix depends on the buoyancy coupling $a$.

In [ ]:
C_velocity = np.array([[0.0, 1.0]])
O_velocity = np.vstack([C_velocity, C_velocity @ A])

A_without_depth_coupling = np.array([
    [0.0, -1.0],
    [0.0, 0.0],
])
O_velocity_without_coupling = np.vstack([
    C_velocity,
    C_velocity @ A_without_depth_coupling,
])

print("Velocity-output observability matrix with buoyancy coupling:")
print(O_velocity)
print("Rank:", np.linalg.matrix_rank(O_velocity))
print()
print("Velocity-output matrix without depth coupling:")
print(O_velocity_without_coupling)
print("Rank:", np.linalg.matrix_rank(O_velocity_without_coupling))

With $a\neq0$, velocity history is affected by depth-dependent buoyancy and reveals both states. When $a=0$, velocity evolution contains no information about absolute depth deviation and the rank falls.

Observability therefore depends on both the chosen output and the physical coupling in the model.

## Exercises

### 1. Change the depth-noise level

Repeat the experiment for depth-noise standard deviations of $0.005$, $0.02$, and $0.05\ \mathrm{m}$. Compare differentiation and observer velocity RMSE.

In [ ]:
# Your code here

### 2. Change the observer poles

Choose two new stable pole pairs. Verify the achieved eigenvalues of $A-LC$, then compare convergence and noise sensitivity.

In [ ]:
# Your code here

### 3. Use a wrong initial estimate

Increase the initial depth and velocity estimation errors. Compare how the slow, nominal, and fast observers remove the error.

In [ ]:
# Your code here

### 4. Introduce model mismatch

Generate the true trajectory using a value of $a$ that differs by $10\%$ from the observer model. Plot the estimation errors and explain what changes.

In [ ]:
# Your code here

## Challenge — sampling and estimation quality

Repeat the complete experiment using several sampling periods. Keep the continuous model and physical noise assumption clearly stated. Determine how coarse sampling changes differentiation error, observer convergence, and numerical integration accuracy.

## Summary

- The model state can contain variables that are not measured directly.
- Depth history contains velocity information through $\delta\dot z=-\delta v$.
- The depth-output pair is observable because the observability matrix has rank two.
- Structural observability does not guarantee practical estimation accuracy.
- Direct differentiation amplifies measurement noise.
- A Luenberger observer combines model prediction with innovation-based correction.
- The observer error is governed by $A-LC$ in the ideal deterministic model.
- Correct gain formulas reproduce the requested observer poles.
- Faster observers converge more quickly but are more sensitive to measurement noise.
- Observability depends on the output choice and the physical coupling encoded in $A$.

## Next notebook

Notebook 08 will introduce Laplace transforms and transfer functions, providing a complementary input–output representation of linear dynamics.